In [1]:
# Bibliothèques nécessaires
from google.colab import drive
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Permet d'accéder au données sur drive
drive.mount("/content/drive/")

Mounted at /content/drive/


# Préparation des données

In [2]:
# Chemin où se trouve les données nettoyés
chemin = "/content/drive/MyDrive/Projet IA Big Data Web/"
df = pd.read_csv(chemin+"arbres_clean.csv")

# Enlever les données non utile à l'entrainement
df = df[["haut_tot", "haut_tronc", "tronc_diam", "fk_arb_etat", "fk_stadedev", "fk_port", "fk_pied", "fk_revetement", "age_estim"]].copy()
print(df.head())

print("Dimension du dataset :", df.shape)
print("Information :\n", df.info())

   haut_tot  haut_tronc  tronc_diam fk_arb_etat fk_stadedev fk_port fk_pied  \
0       NaN         NaN         NaN   supprim'e         NaN     NaN     NaN   
1       NaN         NaN         NaN      abattu         NaN     NaN     NaN   
2       NaN         NaN         NaN   supprim'e         NaN     NaN     NaN   
3       NaN         NaN         NaN   supprim'e         NaN     NaN     NaN   
4       NaN         NaN         NaN      abattu         NaN     NaN     NaN   

  fk_revetement  age_estim  
0           NaN        NaN  
1           NaN        NaN  
2           NaN        NaN  
3           NaN        NaN  
4           NaN        NaN  
Dimension du dataset : (11419, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11419 entries, 0 to 11418
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   haut_tot       11249 non-null  float64
 1   haut_tronc     10321 non-null  float64
 2   tronc_diam     11228 non-nul

# Encodage des valeurs catégorielles

In [3]:
# Encodage pour normaliser les colonnes de type "object"
encoder = OrdinalEncoder()
df[df.select_dtypes(include="object").columns] = encoder.fit_transform(
    df.select_dtypes(include="object")
)
# print(encoder.categories_)
# for i, cats in enumerate(encoder.categories_):
#     print(f"Colonne {i}:")
#     for idx, cat in enumerate(cats):
#         print(f"  {cat} -> {idx}")

df.head(20)

,haut_tot,haut_tronc,tronc_diam,fk_arb_etat,fk_stadedev,fk_port,fk_pied,fk_revetement,age_estim
0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
9,6.0,2.0,37.0,1.0,1.0,9.0,3.0,0.0,15.0


In [4]:
# Sauvegarde de l'encoder
joblib.dump(encoder, chemin+"encoder.pkl")

['/content/drive/MyDrive/Projet IA Big Data Web/encoder.pkl']

## Nettoyage des données

In [5]:
# Réalise la corrélation par rapport à l'attribut 'mean_house_value' et aux autres attribut et l'affiche
corr_matrix = df.corr(numeric_only = True)
print(corr_matrix["age_estim"])

# On remarque qu'on peut retire ceux qui sont peu significatif en corrélation
df = df[["haut_tot", "haut_tronc", "tronc_diam", "fk_stadedev", "age_estim"]].copy()

haut_tot         0.587231
haut_tronc       0.515046
tronc_diam       0.759009
fk_arb_etat      0.128996
fk_stadedev     -0.486856
fk_port          0.283706
fk_pied         -0.129537
fk_revetement    0.096525
age_estim        1.000000
Name: age_estim, dtype: float64


In [6]:
# Suppression de la variable recherché age_estim et des données nulles
df = df.dropna(subset=["haut_tot", "haut_tronc", "tronc_diam", "fk_stadedev", "age_estim"])

In [7]:
print("Dimension du dataset :", df.shape)
print("Information :\n", df.info())
# Dans cette configuration, 18% du dataset est supprimé (mais on évite de faussé les résultats de la prédiction des modèles)

Dimension du dataset : (9449, 5)
<class 'pandas.core.frame.DataFrame'>
Index: 9449 entries, 9 to 11416
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   haut_tot     9449 non-null   float64
 1   haut_tronc   9449 non-null   float64
 2   tronc_diam   9449 non-null   float64
 3   fk_stadedev  9449 non-null   float64
 4   age_estim    9449 non-null   float64
dtypes: float64(5)
memory usage: 442.9 KB
Information :
 None


# Séparation des données

In [8]:
# Séparation des données d'entrainement et de test (80% pour l'entrainement, 20% pour le test)
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

In [9]:
train_data = train_set.drop("age_estim", axis=1) # Enlève le label à prédire
print(train_data)
train_labels = train_set["age_estim"].copy() # Récupère le label à prédire
print(train_labels)

test_data = test_set.drop("age_estim", axis=1) # Enlève le label à prédire
test_labels = test_set["age_estim"].copy() # Récupère le label à prédire

      haut_tot  haut_tronc  tronc_diam  fk_stadedev
9105       7.0         2.0        48.0          1.0
5254      18.0         6.0       210.0          0.0
3694      10.0         4.0        55.0          0.0
3595       5.0         3.0       120.0          0.0
6606      13.0         2.5       100.0          0.0
...        ...         ...         ...          ...
6049      10.0         3.0       110.0          0.0
5487       6.0         3.0        50.0          1.0
5686      16.0         1.0       215.0          0.0
919       12.0         3.0       145.0          0.0
8924       7.0         3.0        43.0          1.0

[7559 rows x 4 columns]
9105    15.0
5254    50.0
3694    25.0
3595    30.0
6606    35.0
        ... 
6049    30.0
5487    10.0
5686    35.0
919     50.0
8924    15.0
Name: age_estim, Length: 7559, dtype: float64


In [10]:
# Scaler pour normaliser les données (évite d'avoir de grandes données)
scaler = StandardScaler()
train_data_scaler = scaler.fit_transform(train_data)
test_data_scaler = scaler.transform(test_data)

In [11]:
# Sauvegarde du scaler
joblib.dump(scaler, chemin+"scaler.pkl")

['/content/drive/MyDrive/Projet IA Big Data Web/scaler.pkl']

In [12]:
# Vérification de l'application du remplacement des valeurs nulles
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9449 entries, 9 to 11416
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   haut_tot     9449 non-null   float64
 1   haut_tronc   9449 non-null   float64
 2   tronc_diam   9449 non-null   float64
 3   fk_stadedev  9449 non-null   float64
 4   age_estim    9449 non-null   float64
dtypes: float64(5)
memory usage: 442.9 KB


In [13]:
# Affiche les 10 premiers éléments des données d'apprentissages
train_data.head(10)
train_labels.head(10)

,age_estim
9105,15.0
5254,50.0
3694,25.0
3595,30.0
6606,35.0
10671,25.0
9562,20.0
10643,30.0
11041,15.0
4912,40.0


# Création des différents modèles

## Modèle 1 : LinearRegressionn

In [14]:
# Création de l'objet LinearRegression
reg = LinearRegression()
# Application de la régression linéaire sur les données
reg = reg.fit(X=train_data_scaler, y=train_labels)

In [15]:
# Prédit les valeurs sur la base d'apprentissage
prediction = reg.predict(test_data_scaler)
# Affiche la prédictions
print(prediction)
# Affiche les vrais valeurs
print(test_labels)

[22.86724715 30.44282123 61.29179465 ... 55.26066619 16.10519125
 46.27338539]
3239    20.0
5426    30.0
820     50.0
1545    50.0
1105    50.0
        ... 
927     70.0
2476    50.0
349     80.0
6823    20.0
3402    40.0
Name: age_estim, Length: 1890, dtype: float64


In [20]:
# Calcule des métriques du modèle
mse = mean_squared_error(test_labels, prediction)
rmse = np.sqrt(mean_squared_error(test_labels, prediction))
mae = mean_absolute_error(test_labels, prediction)
r2 = r2_score(test_labels, prediction)
print(f"MSE : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE : {mae:.2f}")
print(f"R2 : {r2:.2f}")

# Explication des métriques utilisés :
## MSE = Erreur quadratique moyenne, moyenne des carrés des erreurs
## RMSE = Racine de l'erreur quadratique moyenne, racine carré du MSE
## MAE = Erreur absolue moyenne, moyenne des valeurs absolues des erreurs
## R2 = Coefficient de détermination, proportion de la variance de la variable
## cible expliquée par le modèle

MSE : 134.72
RMSE : 11.61
MAE : 8.39
R2 : 0.65


In [21]:
# Tableau de comparaison des valeurs vrais et prédit
result = pd.DataFrame({"y_vrai": test_labels, "y_pred": prediction})
display(result.head())

,y_vrai,y_pred
3239,20.0,22.867247
5426,30.0,30.442821
820,50.0,61.291795
1545,50.0,29.380400
1105,50.0,37.817794


## Modèle 2 : DecisionTreeRegressor

In [22]:
# Création de l'objet DecisionTreeRegressor
tree = DecisionTreeRegressor()
# Applique les valeurs d'apprentissage
tree = tree.fit(X=train_data_scaler, y=train_labels)
# Prédit les valeurs
prediction = tree.predict(test_data_scaler)
# Affiche la prédiction
print(prediction)
# Affiche les valeurs réelles
print(test_labels)

[20.         46.         50.         ... 50.         15.71428571
 46.25      ]
3239    20.0
5426    30.0
820     50.0
1545    50.0
1105    50.0
        ... 
927     70.0
2476    50.0
349     80.0
6823    20.0
3402    40.0
Name: age_estim, Length: 1890, dtype: float64


In [23]:
# Calcule des métriques du modèle
mse = mean_squared_error(test_labels, prediction)
rmse = np.sqrt(mean_squared_error(test_labels, prediction))
mae = mean_absolute_error(test_labels, prediction)
r2 = r2_score(test_labels, prediction)
print(f"MSE : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE : {mae:.2f}")
print(f"R2 : {r2:.2f}")

# Explication des métriques utilisés :
## MSE = Erreur quadratique moyenne, moyenne des carrés des erreurs
## RMSE = Racine de l'erreur quadratique moyenne, racine carré du MSE
## MAE = Erreur absolue moyenne, moyenne des valeurs absolues des erreurs
## R2 = Coefficient de détermination, proportion de la variance de la variable
## cible expliquée par le modèle

MSE : 130.96
RMSE : 11.44
MAE : 6.89
R2 : 0.66


In [24]:
# Tableau de comparaison des valeurs vrais et prédit
result = pd.DataFrame({"y_vrai": test_labels, "y_pred": prediction})
display(result.head())

,y_vrai,y_pred
3239,20.0,20.000000
5426,30.0,46.000000
820,50.0,50.000000
1545,50.0,32.222222
1105,50.0,40.000000


## Modèle 3 : RandomForestRegressor

In [25]:
# Création du modèle de RandomForest
model = RandomForestRegressor(max_depth=2, random_state=42, n_jobs=-1)
# Entrainement du modèle
model.fit(train_data_scaler, train_labels)
# Prédiction des valeurs de test
prediction = model.predict(test_data_scaler)
# Affiche la prédiction
print(prediction)
# Affiche les valeurs réelles
print(test_labels)

[33.50078382 33.69671001 42.44248113 ... 42.44248113 13.27618046
 42.44248113]
3239    20.0
5426    30.0
820     50.0
1545    50.0
1105    50.0
        ... 
927     70.0
2476    50.0
349     80.0
6823    20.0
3402    40.0
Name: age_estim, Length: 1890, dtype: float64


In [26]:
# Calcule des métriques du modèle
mse = mean_squared_error(test_labels, prediction)
rmse = np.sqrt(mean_squared_error(test_labels, prediction))
mae = mean_absolute_error(test_labels, prediction)
r2 = r2_score(test_labels, prediction)
print(f"MSE : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE : {mae:.2f}")
print(f"R2 : {r2:.2f}")

# Explication des métriques utilisés :
## MSE = Erreur quadratique moyenne, moyenne des carrés des erreurs
## RMSE = Racine de l'erreur quadratique moyenne, racine carré du MSE
## MAE = Erreur absolue moyenne, moyenne des valeurs absolues des erreurs
## R2 = Coefficient de détermination, proportion de la variance de la variable
## cible expliquée par le modèle

MSE : 153.93
RMSE : 12.41
MAE : 8.98
R2 : 0.60


In [27]:
# Tableau de comparaison des valeurs vrais et prédit
result = pd.DataFrame({"y_vrai": test_labels, "y_pred": prediction})
display(result.head())

,y_vrai,y_pred
3239,20.0,33.500784
5426,30.0,33.696710
820,50.0,42.442481
1545,50.0,33.696710
1105,50.0,42.442481


# GridSearch

## Modèle 1 (pas possible de faire du GridSearchCV car régression classique)

In [28]:
# Sauvegarde modèle
joblib.dump(reg, chemin+"linear_regression.pkl")

['/content/drive/MyDrive/Projet IA Big Data Web/linear_regression.pkl']

## Modèle 2

In [29]:
# Paramètres à tester
param_grid = {
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.8]
}

In [30]:
# Recherche des paramètres optimal
grid = GridSearchCV(tree, param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid.fit(train_data_scaler, train_labels)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'max_features': ['sqrt', 0.8],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5]},
             scoring='neg_root_mean_squared_error')

In [31]:
# Affichage des meilleurs paramètres
print(grid.best_estimator_)
print(grid.best_params_)
print(-grid.best_score_)

DecisionTreeRegressor(max_depth=10, max_features=0.8, min_samples_leaf=2,
                      min_samples_split=5)
{'max_depth': 10, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
10.84375491862627


In [32]:
# Evaluation du meilleur modèle sur test_date_scaler
y_pred = grid.best_estimator_.predict(test_data_scaler)
mse = mean_squared_error(test_labels, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_labels, y_pred)
r2 = r2_score(test_labels, y_pred)
print(f"MSE : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE : {mae:.2f}")
print(f"R2 : {r2:.2f}")

# Explication des métriques utilisés :
## MSE = Erreur quadratique moyenne, moyenne des carrés des erreurs
## RMSE = Racine de l'erreur quadratique moyenne, racine carré du MSE
## MAE = Erreur absolue moyenne, moyenne des valeurs absolues des erreurs
## R2 = Coefficient de détermination, proportion de la variance de la variable
## cible expliquée par le modèle

MSE : 102.28
RMSE : 10.11
MAE : 6.69
R2 : 0.74


In [33]:
# Tableau de comparaison des valeurs vrais et prédit
result = pd.DataFrame({"y_vrai": test_labels, "y_pred": y_pred})
display(result.head())

,y_vrai,y_pred
3239,20.0,20.000000
5426,30.0,37.805369
820,50.0,80.000000
1545,50.0,32.831858
1105,50.0,37.440299


In [34]:
# Sauvegarde du modèle
joblib.dump(grid.best_estimator_, chemin+"decision_tree.pkl")

['/content/drive/MyDrive/Projet IA Big Data Web/decision_tree.pkl']

## Modèle 3

In [35]:
# Paramètres à tester
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.8]
}

In [36]:
# Recherche des paramètres optimal
grid = GridSearchCV(model, param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid.fit(train_data_scaler, train_labels)

GridSearchCV(cv=5,
             estimator=RandomForestRegressor(max_depth=2, n_jobs=-1,
                                             random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'max_features': ['sqrt', 0.8],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='neg_root_mean_squared_error')

In [37]:
# Affichage des meilleurs paramètres
print(grid.best_estimator_)
print(grid.best_params_)
print(-grid.best_score_)

RandomForestRegressor(max_depth=10, max_features='sqrt', n_estimators=200,
                      n_jobs=-1, random_state=42)
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
9.92906099815971


In [38]:
# Evaluation du meilleur modèle sur test_data_scaler
y_pred = grid.best_estimator_.predict(test_data_scaler)
mse = mean_squared_error(test_labels, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_labels, y_pred)
r2 = r2_score(test_labels, y_pred)
print(f"MSE : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE : {mae:.2f}")
print(f"R2 : {r2:.2f}")

# Explication des métriques utilisés :
## MSE = Erreur quadratique moyenne, moyenne des carrés des erreurs
## RMSE = Racine de l'erreur quadratique moyenne, racine carré du MSE
## MAE = Erreur absolue moyenne, moyenne des valeurs absolues des erreurs
## R2 = Coefficient de détermination, proportion de la variance de la variable
## cible expliquée par le modèle

MSE : 87.96
RMSE : 9.38
MAE : 6.56
R2 : 0.77


In [39]:
# Tableau de comparaison des valeurs vrais et prédit
result = pd.DataFrame({"y_vrai": test_labels, "y_pred": y_pred})
display(result.head())

,y_vrai,y_pred
3239,20.0,21.990205
5426,30.0,38.655179
820,50.0,55.886542
1545,50.0,32.685126
1105,50.0,37.933871


In [40]:
# Sauvegarde du modèle
joblib.dump(grid.best_estimator_, chemin+"random_forest.pkl")

['/content/drive/MyDrive/Projet IA Big Data Web/random_forest.pkl']